# ML Experiment Debugger Demo

This notebook demonstrates how to run the environment, inspect observations, and visualize key metrics. It is designed for data scientists and reviewers who want a hands-on view of the task generation, logging, and diagnostic workflow.

In [ ]:
import json
from ml_env.environment import MLDebugEnv
from ml_env.models import Action

env = MLDebugEnv()
observation = env.reset()
print('Task ID:', observation.task_id)
print('Difficulty:', observation.difficulty)
print('Description:', observation.description)
print('Available tools:', observation.available_tools)

In [ ]:
actions = [
    Action(action_type='fetch_config', keys=['lr', 'optimizer', 'dropout', 'class_weights']),
    Action(action_type='fetch_loss_curve', split='val'),
    Action(action_type='fetch_class_metrics', class_id=0),
]

results = []
for action in actions:
    obs, reward, done, info = env.step(action)
    results.append({
        'action': action.model_dump(),
        'observation': obs.model_dump(),
        'reward': reward.model_dump(),
        'done': done,
        'info': info,
    })
    if done:
        break

print(json.dumps(results, indent=2))

## Evaluation and visualization

The following cell shows how to plot the validation loss and accuracy curves from a generated task. Install `matplotlib` if needed with `pip install matplotlib`.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    raise ImportError('Install matplotlib to plot curves: pip install matplotlib')

# Use the last fetched observation if available, otherwise start a new task
task_obs = env._current_task['data']['loss_curve'] if hasattr(env, '_current_task') else None
if task_obs is None:
    raise RuntimeError('No task data available to plot. Run a task first.')

train_curve = task_obs['train']
val_curve = task_obs['val']
epochs = list(range(1, len(train_curve) + 1))

plt.figure(figsize=(8, 4))
plt.plot(epochs, train_curve, marker='o', label='Train Loss')
plt.plot(epochs, val_curve, marker='o', label='Validation Loss')
plt.title('Train vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()
plt.show()

## How to add a new task

1. Add a new task generator in `ml_env/tasks.py` that returns a dictionary with `task_id`, `difficulty`, `description`, `data`, `ground_truth`, and `grader`.
2. Add the task generator to the correct difficulty bucket in `ml_env/environment.py` under `TASK_POOL`.
3. Add or extend grading logic in `ml_env/graders.py` to recognize the diagnosis, fix type, and keywords for the new task.
4. Run the CLI demo or open `/docs` to verify the task appears correctly.

In [ ]:
# Example task generator skeleton for ml_env/tasks.py

def generate_new_task(seed: int = 123) -> dict:
    from random import Random
    rng = Random(seed)
    return {
        'task_id': 'new_task_type',
        'difficulty': 'medium',
        'description': 'Describe the new ML failure mode here.',
        'data': {
            'logs': ['Epoch 01 ...'],
            'config': {'lr': 0.001},
            'loss_curve': {'train': [1.0, 0.9], 'val': [1.1, 1.0]},
        },
        'ground_truth': {
            'bug_type': 'new_bug',
            'root_cause': 'Explain the cause.',
            'valid_fix_keywords': ['fix', 'configuration'],
        },
        'grader': grade_new_task,
    }